### 03 train (edge hiding + compound disjoint)
reads: prep/  
writes: ckpt/best.pt, ckpt/best_cd.pt, ckpt/train_log*.csv

#### setup

In [1]:
import os, json, time, numpy as np, pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score

PREP = 'C:/Users/Nimishavarma/OneDrive/Dokumenty/hti-project/prep'
CKPT = 'C:/Users/Nimishavarma/OneDrive/Dokumenty/hti-project/ckpt'
os.makedirs(CKPT, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


#### load prep data

In [2]:
meta = json.load(open(f'{PREP}/meta.json'))
n_h, n_c, n_t = meta['n_h'], meta['n_c'], meta['n_t']

C_feat = torch.from_numpy(np.load(f'{PREP}/C_feat.npy')).to(device)
T_feat = torch.from_numpy(np.load(f'{PREP}/T_feat.npy')).to(device)

hc = np.load(f'{PREP}/hc_edges.npy')
ct_tr = np.load(f'{PREP}/ct_train.npy')
ct_vp = np.load(f'{PREP}/ct_val_pos.npy')
ct_vn = np.load(f'{PREP}/ct_val_neg.npy')

print(hc.shape, ct_tr.shape, ct_vp.shape, ct_vn.shape)

(2, 5005) (2, 9916) (2, 2124) (2, 21240)


#### model classes

In [3]:
# same classes as 02_model
class RelConv(nn.Module):
    def __init__(self, in_dim, out_dim, n_rel):
        super().__init__()
        self.W_rel  = nn.Parameter(torch.randn(n_rel, in_dim, out_dim) * 0.01)
        self.W_self = nn.Parameter(torch.randn(in_dim, out_dim) * 0.01)
        self.bias   = nn.Parameter(torch.zeros(out_dim))
    def forward(self, x, edge_sets):
        out = x @ self.W_self
        for r, (src, dst) in enumerate(edge_sets):
            if src.numel() == 0: continue
            m = x[src] @ self.W_rel[r]
            agg = torch.zeros(x.size(0), out.size(1), device=x.device, dtype=out.dtype)
            agg.index_add_(0, dst, m)
            deg = torch.zeros(x.size(0), device=x.device)
            deg.index_add_(0, dst, torch.ones_like(dst, dtype=torch.float))
            out = out + agg / deg.clamp(min=1).unsqueeze(1)
        return out + self.bias

class Encoder(nn.Module):
    def __init__(self, n_h, n_c, n_t, d_in_c, d_in_t, d=128, n_rel=4, dropout=0.2):
        super().__init__()
        self.herb_emb = nn.Embedding(n_h, d)
        self.proj_c = nn.Linear(d_in_c, d)
        self.proj_t = nn.Linear(d_in_t, d)
        self.l1 = RelConv(d, d, n_rel)
        self.l2 = RelConv(d, d, n_rel)
        self.drop = nn.Dropout(dropout)
        self.n_h, self.n_c, self.n_t = n_h, n_c, n_t
    def project(self, Cf, Tf):
        return torch.cat([self.herb_emb.weight, F.relu(self.proj_c(Cf)), F.relu(self.proj_t(Tf))], dim=0)
    def forward(self, Cf, Tf, edge_sets):
        x = self.project(Cf, Tf)
        x = F.relu(self.l1(x, edge_sets))
        x = self.drop(x)
        return self.l2(x, edge_sets)

class DistMult(nn.Module):
    def __init__(self, d, n_rel=1):
        super().__init__()
        self.rel = nn.Parameter(torch.randn(n_rel, d) * 0.01)
    def forward(self, hs, ht, r=0):
        return (hs * self.rel[r] * ht).sum(dim=-1)

class Model(nn.Module):
    def __init__(self, n_h, n_c, n_t, d_in_c, d_in_t, d=128):
        super().__init__()
        self.enc = Encoder(n_h, n_c, n_t, d_in_c, d_in_t, d=d, n_rel=4)
        self.dec = DistMult(d, n_rel=1)
    def encode(self, Cf, Tf, esets):
        return self.enc(Cf, Tf, esets)
    def score_ct(self, Z, c_idx, t_idx):
        return self.dec(Z[self.enc.n_h + c_idx], Z[self.enc.n_h + self.enc.n_c + t_idx])

#### graph edges for message passing (train C-T edges only)

In [4]:
# edge sets: 4 relations total
# r=0 herb -> compound, r=1 compound -> herb, r=2 compound -> target, r=3 target -> compound
h_abs = torch.from_numpy(hc[0]).long().to(device)
c_abs_hc = torch.from_numpy(hc[1] + n_h).long().to(device)

c_abs_ct = torch.from_numpy(ct_tr[0] + n_h).long().to(device)
t_abs_ct = torch.from_numpy(ct_tr[1] + n_h + n_c).long().to(device)

esets = [
    (h_abs,     c_abs_hc),   # H -> C
    (c_abs_hc,  h_abs),      # C -> H
    (c_abs_ct,  t_abs_ct),   # C -> T
    (t_abs_ct,  c_abs_ct),   # T -> C
]
for s,d in esets:
    print(s.shape, d.shape)

torch.Size([5005]) torch.Size([5005])
torch.Size([5005]) torch.Size([5005])
torch.Size([9916]) torch.Size([9916])
torch.Size([9916]) torch.Size([9916])


#### model + optimizer

In [5]:
model = Model(n_h, n_c, n_t, C_feat.shape[1], T_feat.shape[1], d=128).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

ct_tr_t = torch.from_numpy(ct_tr).long().to(device)
vp = torch.from_numpy(ct_vp).long().to(device)
vn = torch.from_numpy(ct_vn).long().to(device)

y_val = np.concatenate([np.ones(vp.shape[1]), np.zeros(vn.shape[1])])

#### negative sampling + val pr-auc

In [6]:
def sample_neg(n, n_t_total):
    return torch.randint(0, n_t_total, (n,), device=device)

@torch.no_grad()
def eval_val():
    model.eval()
    Z = model.encode(C_feat, T_feat, esets)
    sp = model.score_ct(Z, vp[0], vp[1]).cpu().numpy()
    sn = model.score_ct(Z, vn[0], vn[1]).cpu().numpy()
    scores = np.concatenate([sp, sn])
    return average_precision_score(y_val, scores)

#### training loop (bpr loss, early stop on val pr-auc)

In [7]:
BATCH = 1024
EPOCHS = 300
PATIENCE = 20
N_TR = ct_tr_t.shape[1]

best_pr = -1
wait = 0
log = []
t0 = time.time()

for ep in range(1, EPOCHS+1):
    model.train()
    perm = torch.randperm(N_TR, device=device)
    losses = []
    for i in range(0, N_TR, BATCH):
        b = perm[i:i+BATCH]
        c_b = ct_tr_t[0, b]
        t_pos = ct_tr_t[1, b]
        t_neg = sample_neg(b.size(0), n_t)

        Z = model.encode(C_feat, T_feat, esets)
        s_pos = model.score_ct(Z, c_b, t_pos)
        s_neg = model.score_ct(Z, c_b, t_neg)
        loss = -F.logsigmoid(s_pos - s_neg).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(loss.item())

    if ep % 2 == 0 or ep == 1:
        pr = eval_val()
        log.append((ep, float(np.mean(losses)), float(pr)))
        print(f'ep {ep:3d}  loss {np.mean(losses):.4f}  val_prauc {pr:.4f}  elapsed {time.time()-t0:.0f}s')
        if pr > best_pr:
            best_pr = pr
            wait = 0
            torch.save(model.state_dict(), f'{CKPT}/best.pt')
        else:
            wait += 1
            if wait >= PATIENCE:
                print('early stop')
                break

pd.DataFrame(log, columns=['epoch','loss','val_prauc']).to_csv(f'{CKPT}/train_log.csv', index=False)
print('best val prauc', best_pr)

ep   1  loss 0.6924  val_prauc 0.1683  elapsed 1s
ep   2  loss 0.6204  val_prauc 0.2013  elapsed 2s
ep   4  loss 0.4113  val_prauc 0.2473  elapsed 2s
ep   6  loss 0.3572  val_prauc 0.2334  elapsed 2s
ep   8  loss 0.3453  val_prauc 0.2427  elapsed 3s
ep  10  loss 0.3311  val_prauc 0.2542  elapsed 3s
ep  12  loss 0.3125  val_prauc 0.2732  elapsed 4s
ep  14  loss 0.2901  val_prauc 0.3275  elapsed 4s
ep  16  loss 0.2593  val_prauc 0.3147  elapsed 4s
ep  18  loss 0.2443  val_prauc 0.3643  elapsed 5s
ep  20  loss 0.2317  val_prauc 0.3664  elapsed 5s
ep  22  loss 0.2120  val_prauc 0.3657  elapsed 5s
ep  24  loss 0.2056  val_prauc 0.3453  elapsed 6s
ep  26  loss 0.1982  val_prauc 0.3606  elapsed 6s
ep  28  loss 0.1901  val_prauc 0.3445  elapsed 6s
ep  30  loss 0.1897  val_prauc 0.3240  elapsed 7s
ep  32  loss 0.1773  val_prauc 0.3513  elapsed 7s
ep  34  loss 0.1748  val_prauc 0.3685  elapsed 7s
ep  36  loss 0.1730  val_prauc 0.3613  elapsed 8s
ep  38  loss 0.1660  val_prauc 0.3259  elapsed 8s


### compound disjoint training

#### compound disjoint data and graph

In [8]:
ct_tr_cd = np.load(f'{PREP}/ct_train_cd.npy')
ct_vp_cd = np.load(f'{PREP}/ct_val_pos_cd.npy')
ct_vn_cd = np.load(f'{PREP}/ct_val_neg_cd.npy')

# graph edges: H-C same, C-T uses only train compounds
c_abs_ct_cd = torch.from_numpy(ct_tr_cd[0] + n_h).long().to(device)
t_abs_ct_cd = torch.from_numpy(ct_tr_cd[1] + n_h + n_c).long().to(device)

esets_cd = [
    (h_abs,        c_abs_hc),
    (c_abs_hc,     h_abs),
    (c_abs_ct_cd,  t_abs_ct_cd),
    (t_abs_ct_cd,  c_abs_ct_cd),
]

ct_tr_cd_t = torch.from_numpy(ct_tr_cd).long().to(device)
vp_cd = torch.from_numpy(ct_vp_cd).long().to(device)
vn_cd = torch.from_numpy(ct_vn_cd).long().to(device)
y_val_cd = np.concatenate([np.ones(vp_cd.shape[1]), np.zeros(vn_cd.shape[1])])

print('cd train edges:', ct_tr_cd_t.shape, 'val pos:', vp_cd.shape, 'val neg:', vn_cd.shape)

cd train edges: torch.Size([2, 9750]) val pos: torch.Size([2, 2172]) val neg: torch.Size([2, 21720])


#### compound disjoint training loop

In [9]:
model_cd = Model(n_h, n_c, n_t, C_feat.shape[1], T_feat.shape[1], d=128).to(device)
opt_cd = torch.optim.Adam(model_cd.parameters(), lr=1e-3, weight_decay=1e-5)

N_CD = ct_tr_cd_t.shape[1]

@torch.no_grad()
def eval_val_cd():
    model_cd.eval()
    Z = model_cd.encode(C_feat, T_feat, esets_cd)
    sp = model_cd.score_ct(Z, vp_cd[0], vp_cd[1]).cpu().numpy()
    sn = model_cd.score_ct(Z, vn_cd[0], vn_cd[1]).cpu().numpy()
    return average_precision_score(y_val_cd, np.concatenate([sp, sn]))

best_pr_cd = -1
wait_cd = 0
log_cd = []
t1 = time.time()

for ep in range(1, EPOCHS+1):
    model_cd.train()
    perm = torch.randperm(N_CD, device=device)
    losses = []
    for i in range(0, N_CD, BATCH):
        b = perm[i:i+BATCH]
        c_b = ct_tr_cd_t[0, b]
        t_pos = ct_tr_cd_t[1, b]
        t_neg = sample_neg(b.size(0), n_t)

        Z = model_cd.encode(C_feat, T_feat, esets_cd)
        s_pos = model_cd.score_ct(Z, c_b, t_pos)
        s_neg = model_cd.score_ct(Z, c_b, t_neg)
        loss = -F.logsigmoid(s_pos - s_neg).mean()

        opt_cd.zero_grad()
        loss.backward()
        opt_cd.step()
        losses.append(loss.item())

    if ep % 2 == 0 or ep == 1:
        pr = eval_val_cd()
        log_cd.append((ep, float(np.mean(losses)), float(pr)))
        print(f'[cd] ep {ep:3d}  loss {np.mean(losses):.4f}  val_prauc {pr:.4f}  elapsed {time.time()-t1:.0f}s')
        if pr > best_pr_cd:
            best_pr_cd = pr
            wait_cd = 0
            torch.save(model_cd.state_dict(), f'{CKPT}/best_cd.pt')
        else:
            wait_cd += 1
            if wait_cd >= PATIENCE:
                print('early stop (cd)')
                break

pd.DataFrame(log_cd, columns=['epoch','loss','val_prauc']).to_csv(f'{CKPT}/train_log_cd.csv', index=False)
print('best cd val prauc', best_pr_cd)

[cd] ep   1  loss 0.6915  val_prauc 0.1333  elapsed 0s
[cd] ep   2  loss 0.5983  val_prauc 0.1372  elapsed 0s
[cd] ep   4  loss 0.4050  val_prauc 0.1487  elapsed 1s
[cd] ep   6  loss 0.3578  val_prauc 0.1593  elapsed 1s
[cd] ep   8  loss 0.3293  val_prauc 0.1883  elapsed 2s
[cd] ep  10  loss 0.2993  val_prauc 0.2147  elapsed 2s
[cd] ep  12  loss 0.2799  val_prauc 0.2227  elapsed 2s
[cd] ep  14  loss 0.2530  val_prauc 0.2201  elapsed 3s
[cd] ep  16  loss 0.2500  val_prauc 0.2232  elapsed 3s
[cd] ep  18  loss 0.2410  val_prauc 0.2460  elapsed 3s
[cd] ep  20  loss 0.2255  val_prauc 0.2483  elapsed 4s
[cd] ep  22  loss 0.2179  val_prauc 0.2675  elapsed 4s
[cd] ep  24  loss 0.2183  val_prauc 0.2586  elapsed 5s
[cd] ep  26  loss 0.1985  val_prauc 0.2865  elapsed 5s
[cd] ep  28  loss 0.1845  val_prauc 0.2678  elapsed 5s
[cd] ep  30  loss 0.1843  val_prauc 0.2462  elapsed 6s
[cd] ep  32  loss 0.1848  val_prauc 0.2480  elapsed 6s
[cd] ep  34  loss 0.1763  val_prauc 0.2607  elapsed 7s
[cd] ep  3